# BiomedGPT CXR Pipeline — GPU Evaluation

Runs the CXR description pipeline at full scale (~3,300 IU X-ray images) on GPU, across
three configurations (generic checkpoint, fine-tuned checkpoint, fine-tuned + sampling),
checking whether the mode-collapse/hallucination issues found on a small CPU sample hold
at scale.

**Runtime:** GPU (Runtime → Change runtime type → T4 is fine).

Set `MAX_IMAGES = 50` for a quick smoke test before the full run (~30-90 min on a T4).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")


## 1. Environment setup
Clones the OFA-Sys transformers fork (adds the `OFAModel`/`OFATokenizer` classes
BiomedGPT checkpoints need) and installs dependencies.

**Note:** we deliberately do *not* `pip install` the fork. Colab ships a real, current
`transformers` (required by `sentence-transformers`); installing our old fork over it
both fights that dependency and — critically — edits to an editable install aren't
visible to the already-running kernel until a runtime restart. Instead we prepend the
fork's source directory to `sys.path`, which shadows the stock package for the rest of
this session without touching installed packages or needing a restart.

In [ ]:
import os

if not os.path.isdir("OFA"):
    !git clone --single-branch --branch feature/add_transformers https://github.com/OFA-Sys/OFA.git
else:
    print("OFA/ already exists, skipping clone.")

!pip install -q pillow torchvision huggingface_hub datasets sacrebleu rouge-score sacremoses


### `huggingface_hub` compatibility shim

Multiple files in this fork (`file_utils.py`, `dynamic_module_utils.py`, and possibly
others) do `from huggingface_hub import HfFolder, Repository, create_repo,
list_repo_files, whoami` at module load time. Newer `huggingface_hub` (what Colab ships)
dropped `HfFolder`/`Repository` (deprecated Hub-management APIs), which crashes the
import entirely -- and since `transformers.configuration_utils` is imported *eagerly*
(unlike model-specific files like `modeling_ofa.py`, which load lazily on first use),
this must run **before** the bare `import transformers` below, not after it. Rather than
patch every file that happens to import these (fragile -- we already hit it in two
different files), we monkey-patch the `huggingface_hub` module itself, so it doesn't
matter how many files read from it. These symbols are only used for `push_to_hub()` and
an auth-token fallback when a public repo lookup 404s -- neither path is exercised by
this project.

In [ ]:
import huggingface_hub

class _RemovedHubAPI:
    def __getattr__(self, name):
        raise RuntimeError("This deprecated huggingface_hub API was removed upstream and is not used by this project.")
    def __call__(self, *a, **kw):
        raise RuntimeError("This deprecated huggingface_hub API was removed upstream and is not used by this project.")

_shim = _RemovedHubAPI()
patched = []
for _name in ["HfFolder", "Repository", "create_repo", "list_repo_files", "whoami"]:
    if not hasattr(huggingface_hub, _name):
        setattr(huggingface_hub, _name, _shim)
        patched.append(_name)
print("shimmed:", patched if patched else "(none needed, all present)")


In [ ]:
import sys

sys.path.insert(0, os.path.abspath("OFA/transformers/src"))
import transformers
print("transformers loaded from:", transformers.__file__)
assert "OFA/transformers/src" in transformers.__file__, "Stock transformers shadowed our fork -- restart the runtime and rerun from the top."


### Patch a known bug in this fork

BiomedGPT-base checkpoints store a `decoder.image_position_idx` buffer of length 1026,
but this fork hardcodes a 768-element fill (giving 1025), causing a size-mismatch error
on load. Verified fix: bump 768 -> 769. (Found and validated during local CPU testing.)

In [ ]:
import re

path = "OFA/transformers/src/transformers/models/ofa/modeling_ofa.py"
with open(path) as f:
    src = f.read()

old = 'image_position_idx = torch.cat([image_position_idx, torch.tensor([1024] * 768)])'
new = 'image_position_idx = torch.cat([image_position_idx, torch.tensor([1024] * 769)])'

if new in src:
    print("Already patched, skipping.")
elif old in src:
    src = src.replace(old, new)
    with open(path, "w") as f:
        f.write(src)
    print("Patched.")
else:
    raise AssertionError("Neither old nor new line found -- fork may have changed, inspect manually")


## 2. Download and convert checkpoints

In [ ]:
from huggingface_hub import snapshot_download

instruct_base_dir = snapshot_download(repo_id="PanaceaAI/instruct-biomedgpt-base", local_dir="checkpoints/instruct-biomedgpt-base")
print("Downloaded instruct-base to", instruct_base_dir)


### Convert the fairseq `iu_xray.pt` checkpoint to HF format

BiomedGPT's IU-X-ray fine-tuned checkpoint ships in fairseq format only. Verified
(locally, via `strict=True` state_dict load) that ~87% of tensors match the HF port's
naming directly; the rest differ only by submodule renames (e.g. `encoder_attn` ->
`cross_attn`). This cell re-derives that conversion from scratch, so it doesn't depend
on any locally-converted file.

In [ ]:
import argparse
import os

os.makedirs("checkpoints/fairseq_iu_xray", exist_ok=True)
ckpt_path = "checkpoints/fairseq_iu_xray/iu_xray.pt"
if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) > 2_000_000_000:
    print("Already downloaded, skipping.")
else:
    !curl -L "https://www.dropbox.com/s/0llg5e77l6nlb2b/iu_xray.pt?dl=1" -o {ckpt_path} --progress-bar


In [ ]:
import re
import shutil
import torch
from transformers import OFAModel

RENAME_RULES = [
    (re.compile(r"^decoder\.layers\.(\d+)\.encoder_attn_layer_norm\."), r"decoder.layers.\1.cross_attn_layer_norm."),
    (re.compile(r"^decoder\.layers\.(\d+)\.encoder_attn\."), r"decoder.layers.\1.cross_attn."),
    (re.compile(r"^decoder\.layers\.(\d+)\.cross_attn_ln\."), r"decoder.layers.\1.cross_attn_mid_layer_norm."),
    (re.compile(r"^decoder\.layers\.(\d+)\.self_attn_ln\."), r"decoder.layers.\1.self_attn_mid_layer_norm."),
    (re.compile(r"^decoder\.layers\.(\d+)\.ffn_layernorm\."), r"decoder.layers.\1.ffn_layer_norm."),
    (re.compile(r"^encoder\.layers\.(\d+)\.attn_ln\."), r"encoder.layers.\1.self_attn_mid_layer_norm."),
    (re.compile(r"^encoder\.layers\.(\d+)\.ffn_layernorm\."), r"encoder.layers.\1.ffn_layer_norm."),
]
DROP_KEYS = {"encoder.version", "decoder.version"}

def convert_state_dict(fs_sd):
    converted = {}
    for key, tensor in fs_sd.items():
        if key in DROP_KEYS:
            continue
        new_key = key
        for pattern, replacement in RENAME_RULES:
            new_key = pattern.sub(replacement, new_key)
        converted[new_key] = tensor
    return converted

torch.serialization.add_safe_globals([argparse.Namespace])
ckpt = torch.load("checkpoints/fairseq_iu_xray/iu_xray.pt", map_location="cpu", weights_only=True)
converted_sd = convert_state_dict(ckpt["model"])

template_model = OFAModel.from_pretrained("checkpoints/instruct-biomedgpt-base", use_cache=False)
template_keys = set(template_model.state_dict().keys())
converted_keys = set(converted_sd.keys())
missing = template_keys - converted_keys
unexpected = converted_keys - template_keys
assert not missing and not unexpected, f"Key mismatch. Missing: {missing}, Unexpected: {unexpected}"
template_model.load_state_dict(converted_sd, strict=True)
print("strict=True load succeeded -- conversion verified.")

import os
out_dir = "checkpoints/iu-xray-finetuned"
os.makedirs(out_dir, exist_ok=True)
for fname in ["config.json", "vocab.json", "merges.txt"]:
    shutil.copy(os.path.join("checkpoints/instruct-biomedgpt-base", fname), os.path.join(out_dir, fname))
torch.save(converted_sd, os.path.join(out_dir, "pytorch_model.bin"))
print("Wrote", out_dir)
del template_model, converted_sd, ckpt


## 3. Captioner (with vocab-restriction fix)

OFA's shared vocabulary packs image-codebook tokens (indices 50265-58456) and bbox-bin tokens (58457-59456) above the real text vocabulary (0-50264). Fairseq's own caption inference masks these out; this HF port's plain `generate()` doesn't, so without an explicit ban the model sometimes emits a stray `<code_NNNN>` token mid-caption (verified locally: 15/15 outputs corrupted without this fix).

In [ ]:
from PIL import Image
from torchvision import transforms
from transformers import OFATokenizer, OFAModel
from transformers.generation_logits_process import LogitsProcessor, LogitsProcessorList
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CAPTION_PROMPT = " what does the image describe?"
TEXT_VOCAB_SIZE = 50265
IMAGE_RESOLUTION = 256
IMAGE_MEAN = IMAGE_STD = [0.5, 0.5, 0.5]

BEAM_PARAMS = {"num_beams": 5, "no_repeat_ngram_size": 3, "max_length": 64}
SAMPLING_PARAMS = {"do_sample": True, "num_beams": 1, "top_p": 0.9, "temperature": 0.7, "no_repeat_ngram_size": 3, "max_length": 64}


class RestrictToTextVocab(LogitsProcessor):
    def __init__(self, text_vocab_size):
        self.text_vocab_size = text_vocab_size

    def __call__(self, input_ids, scores):
        scores[:, self.text_vocab_size:] = -float("inf")
        return scores


class BiomedGPTCaptioner:
    def __init__(self, checkpoint_dir, device=DEVICE, generation_params=None):
        self.device = device
        self.generation_params = generation_params or BEAM_PARAMS
        self.tokenizer = OFATokenizer.from_pretrained(checkpoint_dir)
        self.model = OFAModel.from_pretrained(checkpoint_dir, use_cache=False).to(device).eval()
        self.transform = transforms.Compose([
            lambda image: image.convert("RGB"),
            transforms.Resize((IMAGE_RESOLUTION, IMAGE_RESOLUTION), interpolation=Image.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
        ])
        self.prompt_ids = self.tokenizer([CAPTION_PROMPT], return_tensors="pt").input_ids.to(device)
        self.logits_processor = LogitsProcessorList([RestrictToTextVocab(TEXT_VOCAB_SIZE)])

    def describe(self, image):
        # accepts either a file path or a PIL Image (full test set is kept in memory, not written to disk)
        if isinstance(image, str):
            image = Image.open(image)
        patch_image = self.transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            generated_tokens = self.model.generate(
                self.prompt_ids,
                patch_images=patch_image,
                logits_processor=self.logits_processor,
                **self.generation_params,
            )
        return self.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0].strip()


## 4. Load the full IU X-ray test set

`X-iZhang/IU-Xray-RRG`, `findings_section` config, `test` split (3,307 real CXR images with paired ground-truth radiology findings, no credentialing required unlike MIMIC-CXR).

In [ ]:
from datasets import load_dataset

MAX_IMAGES = 50  # set to None for the full 3,307-image run once a smoke test passes

ds = load_dataset("X-iZhang/IU-Xray-RRG", name="findings_section", split="test")
if MAX_IMAGES is not None:
    ds = ds.select(range(MAX_IMAGES))
print("Loaded", len(ds), "examples")


## 5. Run inference (3 configurations)

In [ ]:
import json
import time
from tqdm.auto import tqdm

def run_batch(checkpoint_dir, generation_params, output_path, label):
    captioner = BiomedGPTCaptioner(checkpoint_dir, generation_params=generation_params)
    results = []
    start = time.time()
    for i, ex in enumerate(tqdm(ds, desc=label)):
        try:
            description = captioner.describe(ex["main_image"])
        except Exception as e:
            print(f"FAILED on row {i}: {e}")
            continue
        results.append({
            "image_id": f"iu_xray_{i:04d}",
            "generated_description": description,
            "ground_truth_report": ex["findings_section"],
            "checkpoint": label,
        })
    elapsed = time.time() - start
    print(f"{label}: {len(results)} results in {elapsed/60:.1f} min")
    with open(output_path, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
    del captioner
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return results


In [ ]:
results_base_beam = run_batch(
    "checkpoints/instruct-biomedgpt-base", BEAM_PARAMS,
    "outputs_base_beam.jsonl", "base-instruct (beam)"
)

In [ ]:
results_finetuned_beam = run_batch(
    "checkpoints/iu-xray-finetuned", BEAM_PARAMS,
    "outputs_finetuned_beam.jsonl", "iu-xray-finetuned (beam)"
)

In [ ]:
torch.manual_seed(42)
results_finetuned_sampling = run_batch(
    "checkpoints/iu-xray-finetuned", SAMPLING_PARAMS,
    "outputs_finetuned_sampling.jsonl", "iu-xray-finetuned (sampling)"
)

## 6. Evaluate

BLEU, ROUGE-L, exact-duplicate rate (the mode-collapse symptom), and a **negation-aware
14-category clinical-finding F1** approximating CheXbert's label schema.

**Important caveat:** this is our own regex + preceding-window negation heuristic over
the 14 standard CheXpert category names, *not* the actual trained CheXbert BERT labeler.
It's a reasonable proxy (and transparent about what it's checking), but if you want the
real CheXbert F1 for a paper/report, that needs the actual `stanfordmlgroup/CheXbert`
checkpoint and labeler script — a natural follow-up once this heuristic version confirms
the pipeline and overall conclusions are sound.

In [ ]:
import re
from collections import Counter
import sacrebleu
from rouge_score import rouge_scorer

# Approximates CheXbert's 14-category schema. NEGATION_CUES is a small preceding-word
# window check (not full clinical negation detection like NegEx) -- will misjudge
# negation scope in more complex sentences.
CHEXPERT_KEYWORDS = {
    "Enlarged Cardiomediastinum": ["mediastinal widening", "widened mediastinum", "mediastinum is enlarged"],
    "Cardiomegaly": ["cardiomegaly", "heart is enlarged", "enlarged heart", "cardiac silhouette is enlarged"],
    "Lung Opacity": ["opacity", "opacities", "opacification"],
    "Lung Lesion": ["lesion", "nodule", "mass"],
    "Edema": ["edema"],
    "Consolidation": ["consolidation"],
    "Pneumonia": ["pneumonia"],
    "Atelectasis": ["atelectasis"],
    "Pneumothorax": ["pneumothorax"],
    "Pleural Effusion": ["pleural effusion", "effusion"],
    "Pleural Other": ["pleural thickening", "pleural scarring"],
    "Fracture": ["fracture"],
    "Support Devices": ["tube", "catheter", "pacemaker", "sternotomy", "clips", "hardware"],
}
NEGATION_CUES = ["no ", "without ", "not ", "free of ", "negative for ", "absence of ", "resolution of ", "resolved "]
NEGATION_WINDOW_CHARS = 40


def find_assertions(text):
    text = text.lower()
    result = {}
    for keyword, variants in CHEXPERT_KEYWORDS.items():
        status = "absent"
        for variant in variants:
            for m in re.finditer(re.escape(variant), text):
                window = text[max(0, m.start() - NEGATION_WINDOW_CHARS):m.start()]
                status = "negative" if any(c in window for c in NEGATION_CUES) else "positive"
                if status == "positive":
                    break
            if status == "positive":
                break
        result[keyword] = status
    # "No Finding": true iff none of the 13 explicit categories are positive
    result["No Finding"] = "positive" if not any(v == "positive" for v in result.values()) else "negative"
    return result


def evaluate(results):
    generated = [r["generated_description"] for r in results]
    truth = [r["ground_truth_report"] for r in results]

    bleu = sacrebleu.corpus_bleu(generated, [truth]).score
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = sum(scorer.score(t, g)["rougeL"].fmeasure for g, t in zip(generated, truth)) / len(generated)

    dup_counts = Counter(generated)
    _, most_common_count = dup_counts.most_common(1)[0]
    dup_rate = most_common_count / len(generated)

    per_category_f1 = {}
    tp_total = fp_total = fn_total = 0
    for category in list(CHEXPERT_KEYWORDS) + ["No Finding"]:
        tp = fp = fn = 0
        for gen_text, truth_text in zip(generated, truth):
            gen_pos = find_assertions(gen_text)[category] == "positive"
            truth_pos = find_assertions(truth_text)[category] == "positive"
            if gen_pos and truth_pos:
                tp += 1
            elif gen_pos and not truth_pos:
                fp += 1
            elif not gen_pos and truth_pos:
                fn += 1
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        per_category_f1[category] = round(f1, 3)
        tp_total += tp; fp_total += fp; fn_total += fn

    micro_precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
    micro_recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) else 0.0

    return {
        "n": len(results),
        "bleu": round(bleu, 2),
        "rouge_l_f1": round(rouge_l, 4),
        "exact_duplicate_rate": round(dup_rate, 3),
        "unique_outputs": len(dup_counts),
        "chexpert_style_micro_f1": round(micro_f1, 3),
        "per_category_f1": per_category_f1,
    }


In [ ]:
import json

all_results = {
    "base-instruct (beam)": results_base_beam,
    "iu-xray-finetuned (beam)": results_finetuned_beam,
    "iu-xray-finetuned (sampling)": results_finetuned_sampling,
}

summaries = {}
for label, results in all_results.items():
    summaries[label] = evaluate(results)
    print(f"=== {label} ===")
    print(json.dumps(summaries[label], indent=2))
    print()


## 7. Decision guide

Based on the numbers above:

- **If `exact_duplicate_rate` stays high (>=~50%) for the fine-tuned checkpoint at full
  scale** — mode collapse is real and systemic, not a small-sample artifact. Next step:
  fine-tune with rebalanced data (oversample abnormal cases, or add a coverage/
  repetition penalty during training) rather than trying more inference tricks.
- **If `chexpert_style_micro_f1` is low across the board but duplication isn't the main
  driver** (i.e. the generic checkpoint's errors dominate) — the bottleneck is model
  capacity/knowledge, not templating. Next step: try the medium/large BiomedGPT
  checkpoints (same conversion process as `iu_xray.pt`, different Dropbox links in
  BiomedGPT's `checkpoints.md`).
- Compare `bleu`/`rouge_l_f1` against `chexpert_style_micro_f1` per checkpoint — if they
  disagree (high text-overlap, low clinical-finding F1, as seen in the 15/100-image CPU
  sample), that's a real signal that text-overlap metrics alone would be misleading for
  this task.

Download the three `outputs_*.jsonl` files (or copy to Drive) before the Colab runtime
recycles — they're only on the ephemeral VM disk right now.

In [ ]:
from google.colab import files
for fname in ["outputs_base_beam.jsonl", "outputs_finetuned_beam.jsonl", "outputs_finetuned_sampling.jsonl"]:
    files.download(fname)
